## **Deprecated**:
### Approach Retrospective: Why the Initial Spatial Map Matching Approach Was Replaced

This initial approach attempted to infer the vehicle's current stop by comparing its GPS coordinates against every stop in the corresponding GTFS trip using diferent statics files provided by MTBA. Distances were calculated using an Equirectangular approximation to convert latitude/longitude differences into approximate meters.

This approach provided useful spatial information, but it had several important limitations. First, it treated the problem as a purely geometric one. The calculation measured straight-line distance between the vehicle and each stop (see third test), but it did not consider the actual road or rail network, the shape of the route, or whether the vehicle could realistically travel from its current position to that stop. A proper network-aware solution would require additional geographic/routing techniques or map-matching engines capable of projecting the GPS position onto the actual route geometry.

Second, selecting the closest stop was effectively an inference about the vehicle's route. Even after introducing `stop_sequence` constraints, the algorithm was still attempting to reconstruct information that should ideally come from the transit feed itself. Geographic proximity alone cannot reliably determine whether a vehicle is approaching one stop, travelling parallel to another part of the route, or operating on a nearby but different section of the network.

The approach was stopped after discovering that GTFS-Realtime `VehiclePosition` already provides `stop_id`, `current_stop_sequence`, and `current_status`. These fields explicitly describe the vehicle's relationship to a stop and provide information that the previous approach was attempting to infer indirectly.

The new approach (see `02_realtime_stop_analysis.ipynb`) will treat GTFS-Realtime as the primary source for the vehicle's current stop state. GTFS-Static will then be used to enrich and validate that information using `stop_times.txt`, `stops.txt`, `routes.txt`, and, where useful, `shapes.txt`. GPS coordinates, bearing, and other spatial fields will still be retained for validation and further spatial analysis, but they will no longer be responsible for determining which stop the vehicle is associated with.

This results in a simpler and more reliable pipeline:

**GTFS-Realtime vehicle state => GTFS-Static enrichment/validation => temporal and spatial analysis.**


### 01. First Test: Capturing and Inspecting Raw Telemetry

This initial cell acts as a sanity check. It connects to the `raw.vehicle-positions` Kafka topic as a temporary consumer and pulls a finite batch of 500 real messages. 

The goal here is not to compute analytics yet, but to observe the raw shape, noise, and cadence of the MBTA data. I extracted the GeoJSON `location` coordinates into flat `lat` and `lon` columns, loaded the batch into a `pandas` DataFrame, and used DuckDB to sort the results chronologically per vehicle. This gives us our first raw glimpse into a vehicle's breadcrumb trail.

In [14]:
import json
import time
import pandas as pd
import duckdb
from confluent_kafka import Consumer, KafkaException

conf = {
    'bootstrap.servers': '127.0.0.1:9092',
    'group.id': f'duckdb-explorer-{int(time.time())}',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': False,
}

consumer = Consumer(conf)
topic = 'raw.vehicle-positions'
consumer.subscribe([topic])

print(f"Collecting data from topic '{topic}'...")

messages = []
TARGET = 20000
MAX_WAIT_SECONDS = 30
start = time.time()

try:
    while len(messages) < TARGET and (time.time() - start) < MAX_WAIT_SECONDS:
        msg = consumer.poll(timeout=1.0)
        if msg is None:
            continue
        if msg.error():
            raise KafkaException(msg.error())

        payload = json.loads(msg.value().decode('utf-8'))

        if 'location' in payload and 'coordinates' in payload['location']:
            payload['lon'] = payload['location']['coordinates'][0]
            payload['lat'] = payload['location']['coordinates'][1]

        messages.append(payload)

except KeyboardInterrupt:
    print("Capture manually stoped")
finally:
    consumer.close()

df_analitics = pd.DataFrame(messages)
print(f"\n{len(df_analitics)} pings stored in Pandas")

query = """
    SELECT 
        vehicle_id,
        trip_id,
        route_id,
        timestamp,
        lat,
        lon
    FROM df_analitics
    ORDER BY vehicle_id, timestamp
"""

df_analitics = duckdb.sql(query).df()
df_analitics.head(10)


3501 pings stored in Pandas


,vehicle_id,trip_id,route_id,timestamp,lat,lon
0,B-548B3C61,77154087,Blue,2026-08-16T05:57:33.000Z,42.369240,-71.039429
1,B-548B3C61,77154087,Blue,2026-08-16T05:57:33.000Z,42.369240,-71.039429
2,B-548B3C61,77154087,Blue,2026-08-16T05:58:35.000Z,42.370220,-71.038498
3,B-548B3C61,77154087,Blue,2026-08-16T05:58:35.000Z,42.370220,-71.038498
4,B-548B3C61,77154087,Blue,2026-08-16T05:59:03.000Z,42.372311,-71.035538
5,B-548B3C61,77154087,Blue,2026-08-16T05:59:03.000Z,42.372311,-71.035538
6,B-548B3C61,77154087,Blue,2026-08-16T05:59:33.000Z,42.372768,-71.031662
7,B-548B3C61,77154087,Blue,2026-08-16T05:59:52.000Z,42.374352,-71.030167
8,B-548B3C61,77154087,Blue,2026-08-16T05:59:52.000Z,42.374352,-71.030167
9,B-548B3C61,77154087,Blue,2026-08-16T05:59:52.000Z,42.374352,-71.030167


### 02. Second Test: Deduplication and Real Update Frequency (Gaps)

After inspecting the initial raw batch, I noticed duplicate consecutive pings for the same vehicle (identical timestamps and coordinates). This happens because our ingestion service polls the MBTA feed (every 10-15 seconds) faster than the actual vehicles update their internal GPS.

To compute accurate metrics later on, we must filter out this noise. This cell uses DuckDB to deduplicate consecutive identical records. Then, it applies the `LAG()` window function to calculate the actual time gap (in seconds) between genuine vehicle movements, revealing the true update frequency of our telemetry stream.

In [15]:
query_gaps = """
    WITH deduplicated_pings AS (
        -- Deduplicate identical telemetry pings to prevent zero-second gaps
        SELECT 
            vehicle_id,
            trip_id,
            route_id,
            CAST(timestamp AS TIMESTAMPTZ) AS timestamp,
            lat,
            lon
        FROM df_analitics
        GROUP BY vehicle_id, trip_id, route_id, timestamp, lat, lon
    )
    
    -- Compute time deltas between consecutive pings partitioned by vehicle
    SELECT 
        vehicle_id,
        timestamp,
        trip_id,
        route_id,
        lat,
        lon,
        LAG(timestamp) OVER (PARTITION BY vehicle_id ORDER BY timestamp) AS prev_timestamp,
        date_diff('second', 
            LAG(timestamp) OVER (PARTITION BY vehicle_id ORDER BY timestamp), 
            timestamp
        ) AS real_update_gap_seconds
    FROM deduplicated_pings
    ORDER BY vehicle_id, timestamp
"""

df_gaps = duckdb.sql(query_gaps).df()
display(df_gaps.head(15))

,vehicle_id,timestamp,trip_id,route_id,lat,lon,prev_timestamp,real_update_gap_seconds
0,B-548B3C61,2026-08-15 23:57:33-06:00,77154087,Blue,42.369240,-71.039429,NaT,<NA>
1,B-548B3C61,2026-08-15 23:58:35-06:00,77154087,Blue,42.370220,-71.038498,2026-08-15 23:57:33-06:00,62
2,B-548B3C61,2026-08-15 23:59:03-06:00,77154087,Blue,42.372311,-71.035538,2026-08-15 23:58:35-06:00,28
3,B-548B3C61,2026-08-15 23:59:33-06:00,77154087,Blue,42.372768,-71.031662,2026-08-15 23:59:03-06:00,30
4,B-548B3C61,2026-08-15 23:59:52-06:00,77154087,Blue,42.374352,-71.030167,2026-08-15 23:59:33-06:00,19
5,B-548B3C61,2026-08-16 00:00:55-06:00,77154087,Blue,42.376701,-71.027473,2026-08-15 23:59:52-06:00,63
6,B-548B3C61,2026-08-16 00:01:08-06:00,77154087,Blue,42.377972,-71.025940,2026-08-16 00:00:55-06:00,13
7,B-548B3C61,2026-08-16 00:01:36-06:00,77154087,Blue,42.379681,-71.022758,2026-08-16 00:01:08-06:00,28
8,B-548B3C61,2026-08-16 00:02:26-06:00,77154087,Blue,42.380230,-71.019150,2026-08-16 00:01:36-06:00,50
9,B-548B3C61,2026-08-16 00:02:32-06:00,77154087,Blue,42.380489,-71.016998,2026-08-16 00:02:26-06:00,6


#### 02.5. Gap Distribution & Silent Vehicle Thresholds

To build a reliable "stale vehicle" detection system, it must be established a baseline for normal update frequencies. This cell calculates the summary statistics and quantiles for `real_update_gap_seconds` metric. 

The data reveals that while the median update gap is 16 seconds, the 99th percentile sits at 76 seconds. This provides a data-driven threshold: if a vehicle goes silent for more than ~90-120 seconds, we can confidently flag it as a feed interruption, a "silent vehicle", or an out-of-service anomaly, rather than normal network jitter.

In [16]:
df_gaps['real_update_gap_seconds'].describe()

count       1995.0
mean     23.727318
std       37.86031
min            0.0
25%           13.0
50%           16.0
75%           19.0
max          536.0
Name: real_update_gap_seconds, dtype: Float64

In [17]:
df_gaps['real_update_gap_seconds'].quantile([0.5, 0.75, 0.9, 0.95, 0.99])

0.50     16
0.75     19
0.90     33
0.95     60
0.99    300
Name: real_update_gap_seconds, dtype: Int64

### 03. Third Test: Spatial Map Matching against GTFS-Static

Now that we have a clean, deduplicated sequence of vehicle pings, we need to anchor these raw GPS coordinates to the physical transit network. A raw latitude/longitude pair is much more useful once we can relate it to the sequence of stops defined by the vehicle's current GTFS trip.

This cell performs a spatial join between our real-time telemetry (`df_gaps`) and the MBTA's GTFS-static schedule (`stop_times.txt`, `stops.txt`, and `routes.txt`). For each telemetry ping, we retrieve the stops belonging to its specific `trip_id` and calculate the geographic distance between the vehicle and each of those candidate stops. 

At this stage, however, we **do not select the closest stop yet**. The previous implementation used a `QUALIFY ROW_NUMBER()` clause to immediately keep only the single closest stop for each `(vehicle_id, timestamp)`:

```sql
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY vehicle_id, timestamp
    ORDER BY dist_sq_meters ASC
) = 1
```

This approach worked as a basic nearest-neighbor lookup, but it introduced an important limitation: **it considered each GPS ping independently**. The selected stop was determined solely by physical distance, with no knowledge of which stop had been matched in the previous ping.

That was creating a potential problem because GTFS provides an ordered `stop_sequence` for every trip. A vehicle traveling along a trip should generally progress through that sequence rather than jumping arbitrarily between stops based only on which one happens to be geographically closest ($stop\_sequence[t] >= stop\_sequence[t-1]$). For example, if a vehicle is physically closer to stop 20 than stop 10, that does not necessarily mean it has already reached stop 20 if the previous ping was still associated with stop 10, because the vehicle could be at the stop 10 for x seconds picking up passengers.

For that reason, the `QUALIFY` step was removed from this stage. The query now returns **all candidate stops and their corresponding distances for every GPS ping**. This preserves the information needed for the next stage, where the matching process will become sequential rather than purely spatial.

The new flow is therefore:

**GPS ping → candidate stops for the trip → distance calculation → sequence-aware matching**

### Geospatial Correction & Metric Conversion

Originally, I was using a calculation based directly on latitude and longitude differences:

$$
d = \sqrt{
(\Delta \text{lat})^2 + 
(\Delta \text{lon})^2
}
$$

This had an important problem: differences in latitude and longitude are expressed in degrees, but a degree of latitude and a degree of longitude do not represent the same physical distance.

To convert the coordinates into an approximate local metric system, I use the Equirectangular approximation:

$$
d =
\sqrt{
(\Delta \text{lat} \times 111320)^2 +
(\Delta \text{lon} \times 111320 \times \cos(\text{lat}))^2
}
$$

where `111320` approximates the number of meters represented by one degree of latitude.

The cosine correction is applied to longitude because the physical length represented by one degree of longitude decreases with latitude. This produces a distance expressed directly in **meters**, which is much more useful for the proximity and bunching analysis that will follow.The resulting `distance_meters` column therefore represents an approximate physical distance between each vehicle ping and each candidate GTFS stop.

At this stage, a vehicle can have several candidate rows for the same timestamp:

```text
vehicle   timestamp   stop             sequence   distance_meters
1634      13:10:20    Greenbush             0         1439.606
1634      13:10:20    North Scituate       10           11.931
1634      13:10:20    Cohasset             20         1842.552
...
```

This is intentional, since we now retain the complete candidate set so that the next step can combine **spatial proximity** with the **known order of stops** rather than blindly selecting the nearest geographic point.

*Note: Real-world data is messy. During the initial processing, DuckDB produced a `Conversion Error` because it inferred `trip_id` as a numeric type. The MBTA data also contains custom alphanumeric trip IDs, such as `8pmChrisBrownUsher-847359-4763`, created for additional service. To prevent type inference from breaking the query, the relevant GTFS identifiers are explicitly loaded as `VARCHAR` using DuckDB's `types` parameter.*

In [18]:
query_stops_candidates = """
SELECT 
        p.vehicle_id,
        p.trip_id,
        p.timestamp,
        p.route_id,
        p.lat AS ping_lat,
        p.lon AS ping_lon,

        r.route_short_name,
        r.route_long_name,

        st.stop_sequence,
        s.stop_id,
        s.stop_name,


        -- Calculate geospatial distance using the Equirectangular approximation.
        -- Uses 111,320 meters per degree and applies a Cosine correction factor 
        -- on the longitude to account for Earth's curvature at the current latitude.
        SQRT(
            POW(
                (p.lat - s.stop_lat) * 111320,
                2
            ) +
            POW(
                (p.lon - s.stop_lon)
                * 111320
                * COS(RADIANS(p.lat)),
                2
            )
        ) AS distance_meters

    FROM df_gaps AS p

    -- Link telemetry to scheduled trip intervals to find matching stop sequences
    JOIN read_csv_auto(
        '../gtfs_static/MBTA_GTFS/stop_times.txt',
        types={
            'trip_id': 'VARCHAR',
            'stop_id': 'VARCHAR'
        }
    ) AS st
        ON p.trip_id = st.trip_id

    -- Retrieve the exact geographic coordinates (lat/lon) of the scheduled stops
    JOIN read_csv_auto(
        '../gtfs_static/MBTA_GTFS/stops.txt',
        types={
            'stop_id': 'VARCHAR'
        }
    ) AS s
        ON st.stop_id = s.stop_id

    -- Append metadata fields like route names for easier analysis and validation
    JOIN read_csv_auto(
        '../gtfs_static/MBTA_GTFS/routes.txt',
        types={
            'route_id': 'VARCHAR'
        }
    ) AS r
        ON p.route_id = r.route_id

    -- Chronological sort per vehicle and scheduled sequence to track proximity
    ORDER BY
        p.vehicle_id,
        p.trip_id,
        p.timestamp,
        st.stop_sequence

"""

df_candidates = duckdb.sql(query_stops_candidates).df()
display(df_candidates.head(20))

,vehicle_id,trip_id,timestamp,route_id,ping_lat,ping_lon,route_short_name,route_long_name,stop_sequence,stop_id,stop_name,distance_meters
0,B-548B3C61,77154087,2026-08-15 23:57:33-06:00,Blue,42.36924,-71.039429,NaN,Blue Line,1,70038,Bowdoin,2055.703833
1,B-548B3C61,77154087,2026-08-15 23:57:33-06:00,Blue,42.36924,-71.039429,NaN,Blue Line,10,70040,Government Center,1942.882197
2,B-548B3C61,77154087,2026-08-15 23:57:33-06:00,Blue,42.36924,-71.039429,NaN,Blue Line,20,70042,State,1880.954459
3,B-548B3C61,77154087,2026-08-15 23:57:33-06:00,Blue,42.36924,-71.039429,NaN,Blue Line,30,70044,Aquarium,1455.558346
4,B-548B3C61,77154087,2026-08-15 23:57:33-06:00,Blue,42.36924,-71.039429,NaN,Blue Line,40,70046,Maverick,15.840006
5,B-548B3C61,77154087,2026-08-15 23:57:33-06:00,Blue,42.36924,-71.039429,NaN,Blue Line,50,70048,Airport,929.826492
6,B-548B3C61,77154087,2026-08-15 23:57:33-06:00,Blue,42.36924,-71.039429,NaN,Blue Line,60,70050,Wood Island,1795.167423
7,B-548B3C61,77154087,2026-08-15 23:57:33-06:00,Blue,42.36924,-71.039429,NaN,Blue Line,70,70052,Orient Heights,3459.243093
8,B-548B3C61,77154087,2026-08-15 23:57:33-06:00,Blue,42.36924,-71.039429,NaN,Blue Line,80,70054,Suffolk Downs,4221.266183
9,B-548B3C61,77154087,2026-08-15 23:57:33-06:00,Blue,42.36924,-71.039429,NaN,Blue Line,90,70056,Beachmont,4955.321092


### 04. Fourth Test: Sequential Map Matching against GTFS-Static

The previous step gives us the spatial candidates, but choosing the geographically closest stop independently for every GPS ping is not enough, and could cause unexpected and silent problems. The main piece of information that has not yet been used is the ordered `stop_sequence` defined by GTFS, which will allow me to determine which is the real following stop, since every trip contains a predefined sequence of stops. Once a vehicle has been matched to a stop, the next GPS ping should normally remain at that stop (case mentioned earlier, or even if the vehicle is at the main station waiting for the next departure time) or move forward through the sequence, rather than arbitrarily jumping back to an earlier stop caused by, for example, cases where the previous stop is closer than the next one.

The following function introduces that temporal and sequential context. This function processes the candidate stops vehicle by vehicle and trip by trip, in chronological order. This is important because the result of one match becomes information that can be used when evaluating the next GPS ping.

The process goes like this:

1. Sort the observations chronologically. 

In order to guarantee that pings from the same vehicle and trip are evaluated in the same order in which they occurred, the candidate df is first ordered by:

```text
vehicle_id → trip_id → timestamp
```

2. Keep track of the previously matched stop

The function maintains a variable called:

```python
previous_stop_sequence
```

which represents the `stop_sequence` assigned to the previous GPS ping. Initially there is no previous stop, so the first ping has no sequential restriction.

3. Retrieve the candidate stops for the current GPS ping

For each timestamp, the function starts with all the GTFS stops belonging to that trip and their previously calculated `distance_meters`. At this point, the function knows:

```text
Current GPS ping
      +
All stops belonging to this trip
      +
Distance from the ping to every candidate stop
```

4. Apply the sequence constraint

Once a previous stop has been matched, the candidate set is filtered using:

```python
stop_sequence >= previous_stop_sequence
```

This means that the vehicle can:

* remain at the same stop for an undetermined amount of time (resting, awaiting, picking up people);
* move to a later stop;

but it cannot move backward to an earlier stop in the trip sequence. For example, if the previous match was:

```text
stop_sequence = 20
```

then candidates such as:

```text
10 - Invalid
20 - Valid
30 - Valid
40 - Valid
```

are considered, while stop 10 is discarded even if the vehicle happens to be physically closer to it. This is the key difference from the previous nearest-neighbor implementation: distance alone no longer determines the match.

5. Select the closest valid candidate

After the sequence constraint has been applied, the function selects the candidate with the smallest `distance_meters`. Conceptually:

```text
all candidate stops
        ↓
remove stops that violate stop_sequence
        ↓
choose minimum distance
        ↓
current matched stop
```

This combines the two pieces of information we care about:

```text
Spatial information: Which valid stop is physically closest?
Sequential information: Which stops are logically possible (Not going back to the previous stop)?
```

6. Update the matching state

Once a stop has been selected, its `stop_sequence` becomes the new:

```python
previous_stop_sequence
```

This value is then used when processing the next GPS ping. For example:

```text
Ping 1 => stop 10
Ping 2 => stop 10
Ping 3 => stop 10
Ping 4 => stop 20
Ping 5 => stop 20
Ping 6 => stop 30
```

The algorithm therefore maintains a progression such as:

```text
10 => 10 => 10 => 20 => 20 => 30
```

rather than independently choosing a stop for every observation. 

7. Remaining at the same stop is allowed

The constraint intentionally uses:

```python
stop_sequence >= previous_stop_sequence
```

rather than:

```python
stop_sequence > previous_stop_sequence
```

This distinction is important. A vehicle may produce multiple GPS pings while it is still stopped at a station. Those observations should all remain associated with the same `stop_sequence`, so for example, the following are valid observations, and should not be forcer to move to the following stop just because a new ping was produced:

```text
12:34:10 => stop 20
12:34:25 => stop 20
12:34:40 => stop 20
12:34:55 => stop 20
```

8. Preserve unmatched observations

If the sequence constraint eliminates every candidate for a particular ping, the function keeps the observation but marks it as unmatched rather than silently dropping it. This is useful because an unmatched ping is itself valuable information. It may indicate:

* GPS noise;
* an incorrect trip assignment;
* missing GTFS data;
* an unusual vehicle movement;
* or a limitation in the current matching rules.

The output therefore contains fields such as:

```text
matched
previous_stop_sequence
stop_sequence
distance_meters
sequence_advanced
```

which allow the matching process to be inspected rather than treated as a black box. The resulting dataset represents a stateful spatial match:

```text
GPS ping
   ↓
Candidate GTFS stops
   ↓
Distance in meters
   ↓
Previous matched stop
   ↓
stop_sequence >= previous_stop_sequence
   ↓
Closest valid candidate
   ↓
Current matched stop
```

This is the first version of the map-matching logic that uses both the physical position of the vehicle and the known order of the GTFS trip. The purpose of this stage is not yet to produce the final production-quality matcher. Instead, it gives us a controlled baseline that we can inspect against the real MBTA data. The next step will be to evaluate its results and determine whether the simple monotonic sequence constraint is sufficient or whether additional rules are necessary to handle skipped stops, GPS noise, long gaps between observations, or other real-world cases.

In [19]:
df_candidates.columns

Index(['vehicle_id', 'trip_id', 'timestamp', 'route_id', 'ping_lat',
       'ping_lon', 'route_short_name', 'route_long_name', 'stop_sequence',
       'stop_id', 'stop_name', 'distance_meters'],
      dtype='str')

In [20]:
"""
    Match each GPS ping to the closest valid GTFS stop while enforcing
    monotonic stop_sequence progression within each vehicle/trip.

    Main rule:
        current stop_sequence >= previous matched stop_sequence
"""
def sequential_map_match(df_candidates: pd.DataFrame) -> pd.DataFrame:

    df = df_candidates.copy()

    df = df.sort_values(
        ['vehicle_id', 'trip_id', 'timestamp']
    ).reset_index(drop=True)

    matched_rows = []

    for (vehicle_id, trip_id), trip_df in df.groupby(
        ['vehicle_id', 'trip_id'],
        sort=False
    ):
        previous_stop_sequence = None

        for timestamp, ping_df in trip_df.groupby(
            'timestamp',
            sort=True
        ):
            # 1. Candidate stops for this specific GPS ping
            candidates = ping_df.copy()

            # 2. Apply sequence constraint
            if previous_stop_sequence is not None:
                candidates = candidates[
                    candidates['stop_sequence'] >= previous_stop_sequence
                ]

            # 3. If no valid candidate remains, keep the ping unmatched
            if candidates.empty:
                matched_rows.append({
                    'vehicle_id': vehicle_id,
                    'trip_id': trip_id,
                    'route_id': ping_df['route_id'].iloc[0],
                    'timestamp': timestamp,
                    'ping_lat': ping_df['ping_lat'].iloc[0],
                    'ping_lon': ping_df['ping_lon'].iloc[0],
                    'route_short_name': ping_df['route_short_name'].iloc[0],
                    'route_long_name': ping_df['route_long_name'].iloc[0],

                    'closest_stop': None,
                    'stop_id': None,
                    'stop_sequence': None,
                    'distance_meters': None,

                    'previous_stop_sequence': previous_stop_sequence,
                    'sequence_advanced': False,
                    'matched': False,
                })

                continue

            # 4. Among valid candidates, select the closest stop
            best_match = candidates.loc[
                candidates['distance_meters'].idxmin()
            ]

            current_stop_sequence = best_match['stop_sequence']

            # 5. Store result
            matched_rows.append({
                'vehicle_id': vehicle_id,
                'trip_id': trip_id,
                'route_id': best_match['route_id'],
                'timestamp': timestamp,
                'ping_lat': best_match['ping_lat'],
                'ping_lon': best_match['ping_lon'],
                'route_short_name': best_match['route_short_name'],
                'route_long_name': best_match['route_long_name'],

                'closest_stop': best_match['stop_name'],
                'stop_id': best_match['stop_id'],
                'stop_sequence': current_stop_sequence,
                'distance_meters': best_match['distance_meters'],

                'previous_stop_sequence': previous_stop_sequence,

                'sequence_advanced': (
                    previous_stop_sequence is not None
                    and current_stop_sequence > previous_stop_sequence
                ),

                'matched': True,
            })

            # 6. This becomes the state for the next ping
            previous_stop_sequence = current_stop_sequence

    return pd.DataFrame(matched_rows)

In [22]:
df_mapped = sequential_map_match(df_candidates)

display(
    df_mapped[
        [
            'vehicle_id',
            'trip_id',
            'timestamp',
            'closest_stop',
            'stop_sequence', # Stop order number for this route 
            'previous_stop_sequence', # Stop number this vehicle had in the previous ping. NaN if this is the first ping registed
            'sequence_advanced', # True if the vehicle arrived at a new stop
            'distance_meters', # Distance from the vehicle to the following stop in straight line 
        ]
    ].head(50)
)

,vehicle_id,trip_id,timestamp,closest_stop,stop_sequence,previous_stop_sequence,sequence_advanced,distance_meters
0,B-548B3C61,77154087,2026-08-15 23:57:33-06:00,Maverick,40,NaN,False,15.840006
1,B-548B3C61,77154087,2026-08-15 23:58:35-06:00,Maverick,40,40.0,False,149.120956
2,B-548B3C61,77154087,2026-08-15 23:59:03-06:00,Airport,50,40.0,True,475.465806
3,B-548B3C61,77154087,2026-08-15 23:59:33-06:00,Airport,50,50.0,False,196.219213
4,B-548B3C61,77154087,2026-08-15 23:59:52-06:00,Airport,50,50.0,False,21.260245
5,B-548B3C61,77154087,2026-08-16 00:00:55-06:00,Airport,50,50.0,False,362.575792
6,B-548B3C61,77154087,2026-08-16 00:01:08-06:00,Wood Island,60,50.0,True,321.219128
7,B-548B3C61,77154087,2026-08-16 00:01:36-06:00,Wood Island,60,60.0,False,2.283483
8,B-548B3C61,77154087,2026-08-16 00:02:26-06:00,Wood Island,60,60.0,False,305.173008
9,B-548B3C61,77154087,2026-08-16 00:02:32-06:00,Wood Island,60,60.0,False,484.323691


## Explanation of the previous table

The following information were taken from real data

### Case Study 1: Vehicle Advancing to a New Stop

| vehicle_id | trip_id | timestamp | closest_stop | stop_sequence | previous_stop_sequence | sequence_advanced | distance_meters |
|---|---|---|---|---|---|---|---|
| B-548B3C5E | 77154084 | 22:59:21 | Airport | 60 | NaN | False | 310.58 |
| B-548B3C5E | 77154084 | 23:00:28 | Maverick | 70 | 60.0 | True | 14.42 |

#### Analysis & Interpretation:

* At 22:59:21 (Row 9): This is the first recorded ping for this trip (previous_stop_sequence is NaN). The vehicle is 310.58 meters away from the Airport station (stop_sequence 60).
* At 23:00:28 (Row 10): About one minute later, the vehicle moved. The algorithm computed that the closest valid stop is now Maverick (stop_sequence 70), and the vehicle is practically arriving at the platform (14.42 meters away).
* The Logic: Because stop_sequence 70 is greater than 60, the algorithm sets sequence_advanced to True. This confirms the vehicle successfully progressed along its scheduled route.

------------------------------
### Case Study 2: Vehicle Transit / Leaving a Station

| vehicle_id | trip_id | timestamp | closest_stop | stop_sequence | previous_stop_sequence | sequence_advanced | distance_meters |
|---|---|---|---|---|---|---|---|
| O-548B3BA4 | 78493098 | 23:00:18 | Oak Grove | 1 | NaN | False | 206.18 |
| O-548B3BA4 | 78493098 | 23:00:31 | Oak Grove | 1 | 1.0 | False | 346.57 |
| O-548B3BA4 | 78493098 | 23:00:42 | Oak Grove | 1 | 1.0 | False | 621.19 |

#### Analysis & Interpretation:

* The Sequence: The algorithm captures three consecutive pings within a 24-second window. Throughout all of them, the assigned stop remains Oak Grove (stop_sequence 1).
* The Distance Trend: Notice how the distance_meters steadily increases from 206m $\rightarrow$ 346m $\rightarrow$ 621m.
* The Logic: The vehicle is actively driving away from its initial station. However, because it hasn't crossed the proximity threshold to become closer to station number 2 yet, the algorithm locks the sequence at 1 and marks sequence_advanced as False.